# SQL Provider Scoreboard

In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect('../data/raw/mte_patient_intelligence.db')

### Q1 - Sanity Check

In [3]:
q1 = """
SELECT 
    pj.patient_id,
    pj.provider_id,
    pj.treatment_completed,
    pm.region,
    pm.specialty_area
FROM patient_journey pj
JOIN provider_master pm ON pj.provider_id = pm.provider_id
LIMIT 10;
"""
pd.read_sql(q1, conn)

,patient_id,provider_id,treatment_completed,region,specialty_area
0,MTE-PAT-00001,PVD-022,0,Southeast Asia,Transplant Evaluation
1,MTE-PAT-00002,PVD-022,1,Southeast Asia,Transplant Evaluation
2,MTE-PAT-00003,PVD-008,1,West Africa,Transplant Evaluation
3,MTE-PAT-00004,PVD-005,0,East Africa,Orthopedics
4,MTE-PAT-00005,PVD-009,0,East Africa,Fertility
5,MTE-PAT-00006,PVD-022,1,Southeast Asia,Transplant Evaluation
6,MTE-PAT-00007,PVD-003,1,West Africa,Dental
7,MTE-PAT-00008,PVD-002,0,Middle East,Fertility
8,MTE-PAT-00009,PVD-004,1,Southeast Asia,Orthopedics
9,MTE-PAT-00010,PVD-001,1,South Asia,Cardiology


### Q2 - Provider conversion rates

In [4]:
q2 = """
SELECT 
    provider_id,
    COUNT(*) AS total_patients,
    ROUND(AVG(consultation_booked) * 100, 2) AS consultation_rate_pct,
    ROUND(AVG(treatment_completed) * 100, 2) AS treatment_rate_pct
FROM patient_journey
GROUP BY provider_id
ORDER BY treatment_rate_pct DESC;
"""
pd.read_sql(q2, conn)

,provider_id,total_patients,consultation_rate_pct,treatment_rate_pct
0,PVD-020,55,89.09,52.73
1,PVD-007,50,72.00,52.00
2,PVD-008,55,81.82,50.91
3,PVD-001,55,72.73,47.27
4,PVD-011,49,75.51,46.94
5,PVD-017,43,74.42,46.51
6,PVD-015,56,67.86,46.43
7,PVD-003,59,71.19,44.07
8,PVD-022,47,65.96,42.55
9,PVD-021,51,76.47,41.18


### Q3 - Country-level conversion

In [6]:
q3 = """SELECT 
    pj.country,
    cr.region,
    COUNT(*) AS total_patients,
    ROUND(AVG(pj.consultation_booked) * 100, 2) AS consultation_rate_pct,
    ROUND(AVG(pj.treatment_completed) * 100, 2) AS treatment_rate_pct
FROM patient_journey pj
JOIN country_reference cr ON pj.country = cr.country
GROUP BY pj.country, cr.region
ORDER BY treatment_rate_pct DESC;
"""
pd.read_sql(q3, conn)

,country,region,total_patients,consultation_rate_pct,treatment_rate_pct
0,Kenya,East Africa,126,63.49,44.44
1,Malaysia,West Africa,130,66.92,41.54
2,UAE,West Africa,143,64.34,39.16
3,Oman,Middle East,130,64.62,37.69
4,Bangladesh,Middle East,136,68.38,37.50
5,Philippines,East Africa,141,60.99,36.88
6,India,Middle East,94,67.02,36.17
7,Nepal,West Africa,133,65.41,35.34
8,Sri Lanka,South Asia,116,68.10,35.34
9,Thailand,East Africa,114,67.54,35.09


### Q4 - Rank providers by treatment rate Within their region

In [9]:
q4 = """
WITH provider_stats AS (
    SELECT 
        pj.provider_id,
        pm.region,
        COUNT(*) AS total_patients,
        AVG(pj.treatment_completed) AS treatment_rate
    FROM patient_journey pj
    JOIN provider_master pm ON pj.provider_id = pm.provider_id
    GROUP BY pj.provider_id, pm.region
)
SELECT 
    provider_id,
    region,
    total_patients,
    ROUND(treatment_rate * 100, 2) AS treatment_rate_pct,
    RANK() OVER (PARTITION BY region ORDER BY treatment_rate DESC) AS region_rank
FROM provider_stats
ORDER BY region, region_rank;
"""
pd.read_sql(q4, conn)

,provider_id,region,total_patients,treatment_rate_pct,region_rank
0,PVD-007,East Africa,50,52.00,1
1,PVD-011,East Africa,49,46.94,2
2,PVD-009,East Africa,56,39.29,3
3,PVD-024,East Africa,52,36.54,4
4,PVD-019,East Africa,47,29.79,5
5,PVD-005,East Africa,48,27.08,6
6,PVD-013,East Africa,50,24.00,7
7,PVD-002,Middle East,42,40.48,1
8,PVD-016,Middle East,52,36.54,2
9,PVD-001,South Asia,55,47.27,1


### Q5 - Monthly cohort conversion

In [10]:
q5 = """
WITH monthly AS (
    SELECT 
        strftime('%Y-%m', inquiry_date) AS cohort_month,
        COUNT(*) AS inquiries,
        SUM(treatment_completed) AS completions
    FROM patient_journey
    GROUP BY cohort_month
)
SELECT 
    cohort_month,
    inquiries,
    completions,
    ROUND(completions * 100.0 / inquiries, 2) AS conversion_rate_pct,
    SUM(inquiries) OVER (ORDER BY cohort_month) AS cumulative_inquiries,
    SUM(completions) OVER (ORDER BY cohort_month) AS cumulative_completions
FROM monthly
ORDER BY cohort_month;
"""
pd.read_sql(q5, conn)

,cohort_month,inquiries,completions,conversion_rate_pct,cumulative_inquiries,cumulative_completions
0,2025-01,132,43,32.58,132,43
1,2025-02,116,42,36.21,248,85
2,2025-03,113,44,38.94,361,129
3,2025-04,122,48,39.34,483,177
4,2025-05,121,47,38.84,604,224
5,2025-06,117,49,41.88,721,273
6,2025-07,154,61,39.61,875,334
7,2025-08,126,46,36.51,1001,380
8,2025-09,130,48,36.92,1131,428
9,2025-10,129,46,35.66,1260,474


### Q6 - SLA breach flag

In [12]:
q6 = """
SELECT 
    pj.patient_id,
    pj.provider_id,
    pj.response_time_hours,
    cr.baseline_response_expectation_hours,
    CASE 
        WHEN pj.response_time_hours > cr.baseline_response_expectation_hours THEN 1
        ELSE 0
    END AS sla_breach,
    pj.consultation_booked,
    pj.treatment_completed
FROM patient_journey pj
JOIN country_reference cr ON pj.country = cr.country;
"""

pd.read_sql(q6, conn)

,patient_id,provider_id,response_time_hours,baseline_response_expectation_hours,sla_breach,consultation_booked,treatment_completed
0,MTE-PAT-00001,PVD-022,16.4,12,1,0,0
1,MTE-PAT-00002,PVD-022,20.8,12,1,1,1
2,MTE-PAT-00003,PVD-008,15.0,8,0,1,1
3,MTE-PAT-00004,PVD-005,33.9,12,1,1,0
4,MTE-PAT-00005,PVD-009,13.2,6,0,0,0
...,...,...,...,...,...,...,...
1495,MTE-PAT-01496,PVD-024,27.1,8,0,1,0
1496,MTE-PAT-01497,PVD-020,0.2,6,0,1,0
1497,MTE-PAT-01498,PVD-022,6.9,6,1,1,1
1498,MTE-PAT-01499,PVD-010,24.1,8,0,0,0


### Q7 - SLA breach vs conversionm

In [14]:
q7 = """
WITH sla_flagged AS (
    SELECT 
        pj.patient_id,
        pj.consultation_booked,
        pj.treatment_completed,
        CASE 
            WHEN pj.response_time_hours > cr.baseline_response_expectation_hours THEN 1
            ELSE 0
        END AS sla_breach
    FROM patient_journey pj
    JOIN country_reference cr ON pj.country = cr.country
)
SELECT 
    sla_breach,
    COUNT(*) AS total_patients,
    ROUND(AVG(consultation_booked) * 100, 2) AS consultation_rate_pct,
    ROUND(AVG(treatment_completed) * 100, 2) AS treatment_rate_pct
FROM sla_flagged
GROUP BY sla_breach;
"""

pd.read_sql(q7, conn)

,sla_breach,total_patients,consultation_rate_pct,treatment_rate_pct
0,0,961,65.76,36.52
1,1,539,65.31,37.66


### Q8 - Full provider scorecard

In [15]:
q8 = """
WITH provider_sla AS (
    SELECT 
        pj.provider_id,
        AVG(CASE WHEN pj.response_time_hours > cr.baseline_response_expectation_hours THEN 1.0 ELSE 0.0 END) AS sla_breach_rate
    FROM patient_journey pj
    JOIN country_reference cr ON pj.country = cr.country
    GROUP BY pj.provider_id
),
provider_outcomes AS (
    SELECT 
        pj.provider_id,
        COUNT(*) AS total_patients,
        AVG(pj.consultation_booked) AS consultation_rate,
        AVG(pj.treatment_completed) AS treatment_rate,
        AVG(pj.follow_up_completed) AS follow_up_rate,
        AVG(pj.satisfaction_score) AS avg_satisfaction,
        AVG(pj.actual_revenue_inr - pj.service_cost_inr) AS avg_margin
    FROM patient_journey pj
    GROUP BY pj.provider_id
)
SELECT 
    po.provider_id,
    pm.region,
    pm.specialty_area,
    po.total_patients,
    ROUND(po.consultation_rate * 100, 2) AS consultation_rate_pct,
    ROUND(po.treatment_rate * 100, 2) AS treatment_rate_pct,
    ROUND(ps.sla_breach_rate * 100, 2) AS sla_breach_rate_pct,
    ROUND(po.follow_up_rate * 100, 2) AS follow_up_rate_pct,
    ROUND(po.avg_satisfaction, 2) AS avg_satisfaction,
    ROUND(po.avg_margin, 2) AS avg_margin_inr
FROM provider_outcomes po
JOIN provider_sla ps ON po.provider_id = ps.provider_id
JOIN provider_master pm ON po.provider_id = pm.provider_id
ORDER BY treatment_rate_pct DESC;
"""

pd.read_sql(q8, conn)

,provider_id,region,specialty_area,total_patients,consultation_rate_pct,treatment_rate_pct,sla_breach_rate_pct,follow_up_rate_pct,avg_satisfaction,avg_margin_inr
0,PVD-020,West Africa,Transplant Evaluation,55,89.09,52.73,25.45,38.18,4.20,106254.89
1,PVD-007,East Africa,Cardiology,50,72.00,52.00,28.00,24.00,3.90,109057.06
2,PVD-008,West Africa,Transplant Evaluation,55,81.82,50.91,36.36,14.55,4.19,111739.38
3,PVD-001,South Asia,Cardiology,55,72.73,47.27,40.00,36.36,3.52,99932.42
4,PVD-011,East Africa,Orthopedics,49,75.51,46.94,36.73,22.45,3.66,77755.71
5,PVD-017,Southeast Asia,Transplant Evaluation,43,74.42,46.51,37.21,9.30,3.62,86134.07
6,PVD-015,West Africa,Bariatric Surgery,56,67.86,46.43,32.14,25.00,3.82,106709.25
7,PVD-003,West Africa,Dental,59,71.19,44.07,32.20,28.81,3.22,79885.68
8,PVD-022,Southeast Asia,Transplant Evaluation,47,65.96,42.55,31.91,17.02,3.12,88392.79
9,PVD-021,West Africa,Bariatric Surgery,51,76.47,41.18,31.37,19.61,3.28,72455.16
